<a href="https://colab.research.google.com/github/anaclaracandidoo/projetodeep-learning/blob/main/05_banco_dados_historico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalação do SQLAlchemy e bibliotecas auxiliares para o backend
!pip install -q sqlalchemy pydantic

In [3]:
from datetime import datetime
from sqlalchemy import Column, DateTime, Integer, String, Text, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

# 1. Configuração do Banco de Dados SQLite
DATABASE_URL = "sqlite:///historico_eduacessivel.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


# 2. Definição da Tabela de Histórico de Materiais
class HistoricoMaterial(Base):
  __tablename__ = "historico_materiais"

  id = Column(Integer, primary_key=True, index=True)
  titulo = Column(String(150), nullable=False)
  texto_original = Column(Text, nullable=False)
  texto_adaptado = Column(Text, nullable=False)
  tipo_adaptacao = Column(String(100), nullable=False)
  data_criacao = Column(DateTime, default=datetime.utcnow)


# Criar as tabelas no banco de dados (Sintaxe Corrigida)
Base.metadata.create_all(bind=engine)


# 3. Funções de CRUD
def salvar_historico(
    db, titulo: str, original: str, adaptado: str, tipo: str
) -> HistoricoMaterial:
  registro = HistoricoMaterial(
      titulo=titulo,
      texto_original=original,
      texto_adaptado=adaptado,
      tipo_adaptacao=tipo,
  )
  db.add(registro)
  db.commit()
  db.refresh(registro)
  return registro


def listar_historicos(db):
  return (
      db.query(HistoricoMaterial)
      .order_by(HistoricoMaterial.data_criacao.desc())
      .all()
  )


# 4. Teste Prático de Inserção e Consulta
db = SessionLocal()

print("--- 1. Inserindo registros de teste no Banco de Dados ---")
m1 = salvar_historico(
    db=db,
    titulo="Aula de Fotossíntese",
    original="A fotossíntese é o processo físico-químico...",
    adaptado="A fotossíntese ocorre nas plantas usando a luz solar...",
    tipo="Linguagem Clara e Simplificada",
)

m2 = salvar_historico(
    db=db,
    titulo="Resumo do Sistema Solar",
    original="O Sistema Solar é constituído pelo Sol...",
    adaptado="O Sistema Solar tem 8 planetas que giram ao redor do Sol.",
    tipo="Resumo em Tópicos",
)

print("Registros salvos com sucesso!")

print("\n--- 2. Consultando o Histórico Registrado ---")
historico = listar_historicos(db)
for item in historico:
  print(
      f"[ID {item.id}] {item.titulo} | Tipo: {item.tipo_adaptacao} | Data:"
      f" {item.data_criacao.strftime('%d/%m/%Y %H:%M')}"
  )
  print(f"   Adaptado: {item.texto_adaptado}\n")

db.close()

--- 1. Inserindo registros de teste no Banco de Dados ---
Registros salvos com sucesso!

--- 2. Consultando o Histórico Registrado ---
[ID 2] Resumo do Sistema Solar | Tipo: Resumo em Tópicos | Data: 01/10/2026 23:33
   Adaptado: O Sistema Solar tem 8 planetas que giram ao redor do Sol.

[ID 1] Aula de Fotossíntese | Tipo: Linguagem Clara e Simplificada | Data: 01/10/2026 23:33
   Adaptado: A fotossíntese ocorre nas plantas usando a luz solar...

